[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
# !echo $GT_TOKEN # makesure the token is loaded

In [4]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [5]:
!pip install -r jrcai_corekit/requirements.txt

/bin/bash: /home/majed_alshaibani/Projects/instructions-tuning/venv/bin/pip: /home/majed_alshaibani/Projects/InstructionsTuning/venv/bin/python3: bad interpreter: No such file or directory


add jrcai_corekit to path

In [6]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Constants

In [8]:
MODEL_PATH = "/hdd/shared_models/AceGPT-7B"
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH
DATASETS_USED_FOR_TUNING = {
    'NLU': 'ArabicMMLU',
    'machine_translation':'opus-100',
    'dialect_identification':'AraBench_dev',
    
    'NLI':'ArEntail',
    # 'summarization':'xlsum',
    'sarcasm_detection':'ArSarcasm_v2',
}

## Finetuning

In [9]:
GLOBAL_SEED = 42

In [10]:
import random
random.seed(GLOBAL_SEED)

### Get the training and validation samples

In [11]:
import json

train_samples,eval_samples = [],[]

for task_name,dataset_name in DATASETS_USED_FOR_TUNING.items():
    train_sampels_path = f'Notebooks/Experiments/{task_name}/{dataset_name}/AceGPT/samples_used_for_tuning/train.json'
    eval_sampels_path = f'Notebooks/Experiments/{task_name}/{dataset_name}/AceGPT/samples_used_for_tuning/val.json'
    dataset_train_samples = json.load(open(train_sampels_path))
    dataset_eval_samples = json.load(open(eval_sampels_path))
    # to balance the tuning dataset
    # dataset_train_samples = dataset_train_samples[:10_000]
    # dataset_eval_samples = dataset_eval_samples[:1_000]
    print(len(dataset_train_samples),len(dataset_eval_samples))
    train_samples.extend(dataset_train_samples)
    eval_samples.extend(dataset_eval_samples)
    
train_samples = list(map(tuple,train_samples))
eval_samples = list(map(tuple,eval_samples))

random.seed(GLOBAL_SEED)
random.shuffle(train_samples)

random.seed(GLOBAL_SEED)
random.shuffle(eval_samples)
    
len(train_samples),len(eval_samples),train_samples[:5], eval_samples[:5]

9000 1000
27000 3000
27000 3000
4500 500
11293 1255


(78793,
 8755,
 [('The following tweet: "تنفي حركة #حماس  أن تكون تلقت دعوة من الاخوة في حركة فتح لحضور :مؤتمر فتح السابع"" is sarcastic? \nTrue or False',
   ' False'),
  ('The following tweet: كونوا على يقين أن هناك شيء ينتظركم بعد الصبر ليبهركم فينسيكم مرارة الألم و الحزن و التضحيات is sarcastic? \nTrue or False',
   ' False'),
  ("You're tasked with identifying wether a given Arabic phrase is MSA or dialectical, and if not MSA, what's the dialect of it.\nYour answer must be one of the following choices: Tunisian, MSA, Morrocan, Qatari, Egyptian, Lebanese.\n---\nSentence:  حينئذ دعا هيرودس ٱلمجوس سرا، وتحقق منهم زمان ٱلنجم ٱلذي ظهر.  \nAnswer:",
   ' MSA'),
  ('You are tasked with solving a multiple-choice question (MCQ). Your goal is to analyze the question, consider all options carefully, and provide the correct answer.\nHere is the MCQ question: ل من اجري سباق بواسطة المركبات الآلية على الطريق دون تصريح او بالمخالفة للتصريح يعاقب\nAnd here are the answer options:\n\nA. الحبس مدة 

## Finetune the LLM

In [12]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [13]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [14]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "/hdd/shared_models/AceGPT-7B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file /hdd/shared_models/AceGPT-7B/pytorch_model.bin
Instantiating LlamaForCausalLM model under default dtype torch.bfl

In [ ]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(r=32, lora_alpha=64),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=4,
    eval_batch_size=4,
    output_dir=f'Notebooks/Experiments/cross_tasks_tuning/tuned_models/{MODEL_NAME}',
    early_stopping_patience=50,
    eval_steps=500,
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=32, target_modules={'q_proj', 'v_proj'}, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 8755
  Batch size = 4


loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "meta-llama/Llama-2-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}



{'eval_loss': 2.4042224884033203, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 214.9301, 'eval_samples_per_second': 40.734, 'eval_steps_per_second': 10.185}


***** Running training *****
  Num examples = 78,793
  Num Epochs = 10
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 1
  Total optimization steps = 196,990
  Number of trainable parameters = 16,777,216


Step,Training Loss,Validation Loss,Model Preparation Time
500,0.634500,0.591801,0.000300



***** Running Evaluation *****
  Num examples = 8755
  Batch size = 4
loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "meta-llama/Llama-2-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}



{'eval_loss': 0.5918011665344238, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 209.1425, 'eval_samples_per_second': 41.861, 'eval_steps_per_second': 10.467, 'epoch': 0.025381999086248032}


In [ ]:
exit()